# SmartScan — HARD tier training on the full corpus

Deepnote project `942718cf-6e64-4d7e-a07a-721b33ec0d98`.

Trains the HARD tier on **all 800 published HARD episodes** rather than the ~40
regenerated from seeds that produced the shipped checkpoints, then scores the
result on the mission metric.

**Read this before interpreting any result.** A rising training return is not
evidence of a better scheduler. `dqn_hard` went from −570 to +312 over 3M steps
and still missed more emitters than a plain sweep. The only number that counts
is *emitters never intercepted*, in the last cell.

**And part of HARD cannot be fixed by training.** AgileBeamRadar offers a median
0.81 expected looks per episode under any uniform-coverage policy — a ~44% floor
miss rate before any scheduler decides anything. CircularScanRadar is the real
target: 0.5% floor against a 22% actual miss rate.


## 1. Environment

In [ ]:
!nvidia-smi || echo "no GPU visible"
!git clone https://github.com/shirish-raj-gupta/SIH26055_Prototype.git 2>/dev/null || echo "already cloned"
%cd SIH26055_Prototype
!pip install -q -e ".[ml,viz]"


In [ ]:
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## 2. Kaggle credentials

Set `KAGGLE_USERNAME` and `KAGGLE_KEY` as Deepnote **environment variables**
(Integrations → Environment variables), not as literals in a cell — this
notebook is in a public repository.


In [ ]:
import os

assert os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"),     "Set KAGGLE_USERNAME and KAGGLE_KEY as Deepnote environment variables first."
print("kaggle user:", os.environ["KAGGLE_USERNAME"])


## 3. Fetch and verify the corpus

Refuses to continue if it silently falls back to regenerating from seeds —
that fallback is the exact sampling this notebook exists to avoid.


In [ ]:
!python scripts/deepnote_hard_train.py --stage data


## 4. Train the predictor on everything

Streams all HARD episodes. `--windows-per-episode` trades RAM for how much of
each episode is used; raise it if the box has headroom.


In [ ]:
!python scripts/deepnote_hard_train.py --stage predictor --windows-per-episode 600


## 5. Train the RL schedulers

3M steps each is what the shipped checkpoints got. Raise `--rl-steps` if you
have the hours — but see the note at the top about what more steps buys.


In [ ]:
!python scripts/deepnote_hard_train.py --stage rl --rl-steps 3000000 --rl ppo,dqn,hybrid


## 6. The only cell that decides whether this worked

Emitters never intercepted, per class, against the baselines and the physical
floor. A trained model counts only if its total is **below** the best baseline.


In [ ]:
!python scripts/deepnote_hard_train.py --stage evaluate --n-seeds 8


## 7. Take the checkpoints home

The shipped checkpoints were copied to `*_shipped.*` before training, so the
comparison survives even if the new ones are worse.


In [ ]:
!ls -la runs/checkpoints/ | grep hard
!cd runs/checkpoints && tar czf /work/hard_checkpoints.tgz *hard*.pt *hard*.json
print("download /work/hard_checkpoints.tgz from the Deepnote file browser")
